In [1]:

import requests
from bs4 import BeautifulSoup
import pandas as pd

# Official FSC statistics webpage
url = "https://stat.fsc.gov.tw/competitivenessinfoen"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("HTTP Status:", response.status_code)
print("Page Title:", soup.title.get_text(strip=True) if soup.title else "N/A")

# Extract visible webpage text
page_text = soup.get_text(" ", strip=True)

print("\nCredit card statistics found:")
for keyword in [
    "Credit cards in circulation",
    "Aggregate credit card debt"
]:
    position = page_text.find(keyword)
    if position >= 0:
        print(page_text[position:position + 150])
    else:
        print(keyword, "- not found in HTML")


HTTP Status: 200
Page Title: Financial Competitiveness Key Information | FSC

Credit card statistics found:
Credit cards in circulation (Unit: 1000 cards) 59,612 +123.00 Aggregate credit card debt (Unit: NT$ billion) 432.59 -21.77 Banks and branch units (per
Aggregate credit card debt (Unit: NT$ billion) 432.59 -21.77 Banks and branch units (per 100,000 population) - - Avg. ratio of Tier I capital to risk-


In [2]:

import re

# Extract indicators and their values from webpage text
indicators = {
    "credit_cards_in_circulation": (
        r"Credit cards in circulation\s*\(Unit:\s*1000 cards\)\s*([\d,]+)"
    ),
    "aggregate_credit_card_debt": (
        r"Aggregate credit card debt\s*\(Unit:\s*NT\$ billion\)\s*([\d,.]+)"
    )
}

records = []

for indicator, pattern in indicators.items():
    match = re.search(pattern, page_text, re.IGNORECASE)

    if match:
        records.append({
            "indicator": indicator,
            "value": float(match.group(1).replace(",", "")),
            "source": "FSC Taiwan",
            "source_url": url
        })

scraping_df = pd.DataFrame(records)

display(scraping_df)

# Export to CSV
scraping_df.to_csv(
    "credit_card_scraping.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Records extracted:", len(scraping_df))
print("CSV exported successfully!")


,indicator,value,source,source_url
0,credit_cards_in_circulation,59612.00,FSC Taiwan,https://stat.fsc.gov.tw/competitivenessinfoen
1,aggregate_credit_card_debt,432.59,FSC Taiwan,https://stat.fsc.gov.tw/competitivenessinfoen


Records extracted: 2
CSV exported successfully!
